<a href="https://colab.research.google.com/github/vincentmartin/tp-information-retrieval-with-llm-student-version/blob/main/1-Recherche%20d'information%20classique.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Partie 1. - Recherche d'Information classique

Dans cette partie, nous allons mettre en œuvre des principes et modèles classiques de Recherche d'Information. Le jeu de données est une collection de livres au format texte (.txt) de Henry Rider Haggard. Jetez un œil à ces documents dans le dossier _data_.

En sortie de ce module, vous serez capable de :

- Construire un index inversé ;
- Effectuer des requêtes simples selon le modèle booléen :
- Calculer la pondération des termes selon la méthode TF-IDF ;
- Mettre en œuvre le modèle vectoriel de recherche d'information et y appliquer des requêtes.

### Import des bibliothèques logicielles et configuration

Les lignes suivantes permettent d'instancier un objet de la classe `IRSystem` représentant notre moteur de recherche et de charger les données en RAM.

In [1]:
import sys

# Vérifie si le code est exécuté sur Google Colab
if 'google.colab' in sys.modules:
    # Commandes à exécuter uniquement sur Google Colab
    !git clone https://github.com/vincentmartin/tp-information-retrieval-with-llm-student-version.git
    %cd tp-information-retrieval-with-llm-student-version
else:
    # Commandes à exécuter si ce n'est pas sur Google Colab
    print("Pas sur Google Colab, ces commandes ne seront pas exécutées.")

Pas sur Google Colab, ces commandes ne seront pas exécutées.


#### Chargement des données

Les lignes ci-dessous permettent de charger les données qui sont un ensemble de 60 livres au format texte (.txt) d'[Henry Rider Haggard](https://fr.wikipedia.org/wiki/Henry_Rider_Haggard).


In [2]:
from classic_ir.IRSystem import *

# !rm -rf ./data/RiderHaggard/stemmed
ir_system = IRSystem()
ir_system.read_data('./data/RiderHaggard') # chargement des données et prétraitement des documents (stemming).

Reading in documents...
Already stemmed!


### Exercice 1. - Construction de l'index inversé

Ce premier exercice a pour objectif de construire l'index inversé non positionnel. L'attribut `self.inverted_index` est un tableau associatif qui associe une liste d'entiers (docIDs) à un mot (word).
 
Documentation ici https://docs.python.org/3/library/collections.html#collections.defaultdict. 

Exercice : modifier la fonction `index` pour calculer l'index inversé.

**Indication** : profitez du parcours des documents pour remplir aussi `self.tf`, qui associe à chaque docID un `Counter` donnant le nombre d'occurrences de chaque mot dans ce document. Vous en aurez besoin à l'exercice 3 pour calculer les poids TF-IDF. 

Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Inverted Index Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [3]:
# Exercice 1. Indexation

def index(self):
    """
    Construit l'index inversé et le stocke dans self.inverted_index.

    Dans le code ci-dessous, seul le dictionnaire des tokens est construit. Les postings lists sont vides.
    """
    print("Indexing...")
    self.tf = defaultdict(Counter) # tf est un dictionnaire qui à un document associe un Counter de mots (à remplir, utilisé à l'exercice 3).
    inverted_index = defaultdict(list) # inverted_index est un dictionnaire qui à un mot associe une liste de documents. 

    #self.docs : les documents sont prétraités, chacun représenté par une liste de mots
    #self.titles : pour retrouver à quel document correspond chaque liste dans self.docs

    #self.tf : le counter : enregistre le nombre d'occurrence de chaque mot dans chaque document


    #parcours des documents
    for doc_id,document in enumerate(self.docs):
        #Compte les occurrences de chaque mot dans ce document
        self.tf[doc_id]=Counter(document)

        # Ajoute le document une seule fois à chaque posting list
        for word in self.tf[doc_id]:
            inverted_index[word].append(doc_id)

    self.inverted_index = inverted_index

# Ne pas modifier les lignes ci-dessous
IRSystem.index = index
ir_system.index()
run_tests(ir_system, part=0)

Indexing...
===== Running tests =====
Inverted Index Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


### Exercice 2. - Recherche booléenne

Ce deuxième exercice a pour objectif d'implémenter la recherche booléenne. La requête `query` est une liste de mots _racinisés_ (stemming de Porter) et l'algorithme doit rechercher les documents qui contiennent TOUS ces mots. 


Exercice : modifier la fonction `boolean_retrieve` pour implémenter la recherche booléenne.


Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Boolean Retrieval Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [4]:
# Exercice 2. Recherche booléenne
def boolean_retrieve(self, query):
    """
    A partir d'une requête sous la forme d'une liste de mots *racinisés*,
    retourne la liste des documents dans lesquels *tous* ces mots apparaissent (ie une requête AND).
    Retourne une liste vide si la requête ne retourne aucun document.

    Dans le code ci-dessous, tous les documents répondent.
    """
    #query : liste des mots racinisés

    # A partir de l'index inversé, on parcourt la query et on cherche les documents de chaque mot racinisés. 
        # on fait l'intersection sur l'ensemble des mots de la query et on retourne le resultat

    #on parcourt la query, on compare l'index inversé du premier élément avec l'élément courant et on conserve l'intersection des 2 ensembles puis on répète
    query_inverted_index=set(self.inverted_index[query[0]])
    for mot in query[1:]:
        query_inverted_index=query_inverted_index.intersection(set(self.inverted_index[mot])) 
        
    """docs = list()
    for doc_id in range(len(self.docs)):
        docs.append(doc_id)
    return docs"""

    return sorted(query_inverted_index)

# Ne pas modifier les lignes ci-dessous
IRSystem.boolean_retrieve = boolean_retrieve
run_tests(ir_system, part=1)

===== Running tests =====
Boolean Retrieval Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


### Exercice 3. - Calcul des poids TF-IDF des termes dans les documents

Dans ce troisième exercice, l'objectif est de pré-calculer les poids TF-IDF pour chaque terme dans chaque document. Pour ce faire, appliquer la formule vue en cours. Utiliser le logarithme en base 10.


Exercice : modifier la fonction `compute_tfidf` pour calculer les poids TF-IDF.

Le résultat ci-dessous indique que vous avez réussi.
```
Calculating tf-idf...
===== Running tests =====
TF-IDF Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [8]:
# Exercice 3. calcul des scores tf-idf
def compute_tfidf(self):
    """
    Calcule les scores tf-idf pour tous les mots de tous les documents et les stocke dans self.tfidf.

    Dans le code ci-dessous, les scores tf-idf sont tous nuls.
    """
    print("Calculating tf-idf...")

    
    self.tfidf = defaultdict(Counter)
    N = len(self.docs)  # N = nombre de documents
    for word in self.vocab:
        #df : nombre de documents contenant word
        df=len(self.inverted_index[word])
        #Inverse document frequency
        idf=np.log10(N/df)
        for i in range(N):
            try:
                #Term Frequency : la fréquence de word dans le document
                tf=1+np.log10(self.tf[i][word])
                self.tfidf[i][word] = tf*idf
            except ValueError:
                self.tfidf[i][word] = 0.

# Ne pas modifier les lignes ci-dessous
IRSystem.compute_tfidf = compute_tfidf
ir_system.compute_tfidf()
run_tests(ir_system, part=2)

Calculating tf-idf...


C:\Users\darks\AppData\Local\Temp\ipykernel_35996\1021195316.py:21: RuntimeWarning: divide by zero encountered in log10
  tf=1+np.log10(self.tf[i][word])


===== Running tests =====
TF-IDF Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000


#### Question - Du bruit dans le corpus ?

Les livres proviennent du [Projet Gutenberg](https://www.gutenberg.org/). Ouvrez quelques fichiers du dossier `data/RiderHaggard/raw` et regardez leur dernière ligne : la plupart se terminent par une mention ajoutée par le Projet Gutenberg, du type « End of the Project Gutenberg EBook of She, by H. Rider Haggard ».

1. Affichez, pour les termes `gutenberg`, `ebook` et `project`, le nombre de documents qui les contiennent (df) et leur idf. Rappel : `self.inverted_index` attend des mots racinisés, utilisez `get_posting_unstemmed`.
2. Pourquoi le poids TF-IDF de `gutenberg` reste-t-il faible dans tous les documents, alors que ce mot n'a rien à voir avec les romans ?
3. Quel serait l'idf de `gutenberg` si un seul livre contenait cette mention ? Quel risque cela poserait-il pour la recherche ?
4. Quel traitement proposeriez-vous avant l'indexation ? Faites le lien avec la partie du cours « Représenter les documents ».

In [ ]:
# Question - Du bruit dans le corpus ?
N = len(ir_system.docs)
for word in ['gutenberg', 'ebook', 'project']:
    # A COMPLETER : afficher df et idf du mot
    #df : nombre de documents contenant word
            df=len(ir_system.get_posting_unstemmed(word))
            #Inverse document frequency
            idf=np.log10(N/df)
            print(df,idf,word)

# Vos réponses aux questions 2 à 4 :
# 2 : Le TF-IDF mesure l'importance d'un mot dans un document par rapport à un ensemble de documents. Comme il est présent dans plusieurs document, son importance est faible.
# 3 : TF-IDF = TF*IDF = 1+np.log10(self.tf[i][word]) * idf=np.log10(N/len(self.inverted_index[word])) = 1+np.log10(self.tf[i][word])*np.log10(N) : Le terme gutenberg deviendrait un terme discriminant.
# 4 : On pourrait pré traiter les documents en supprimant tout contenu comme l'en tête et le texte en fin de livre ne faisant pas parties du contenu du livre.


52 0.062147906748844434 gutenberg
38 0.1983676537668335 ebook
60 0.0 project


### Exercice 4. - Calcul de la similarité cosinus.

Dans ce quatrième exercice, l'objectif est de calculer la similarité cosinus entre le vecteur requête `query` et chaque vecteur document. Pour ce faire, appliquer la formule vue en cours en considérant les éléments suivants :
- **Vecteurs documents** : poids TF-IDF calculés à l'exercice 3, **normalisés L2** (on divise par la norme du vecteur document).
- **Vecteur requête** : poids TF logarithmique `1 + log10(tf)`, **sans IDF et sans normalisation**.
- Utiliser le logarithme en base 10.

Le score attendu est donc :

$$score(q, d) = \frac{\sum_{t \in q} w_{t,q} \cdot w_{t,d}}{\lVert d \rVert_2}$$

Ne pas normaliser le vecteur requête ne change pas le classement (la norme de la requête est la même pour tous les documents), mais change la valeur du score, qui est vérifiée par le test. En notation SMART, il s'agit du schéma **ltc.lnn**.

Exercice : modifier la fonction `rank_retrieve` pour implémenter la similarité cosinus.

Le résultat ci-dessous indique que vous avez réussi.
```
===== Running tests =====
Cosine Similarity Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
```

In [16]:
# Exercice 4. Similarité cosinus
def rank_retrieve(self, query):
    """
    A partir d'une requête (une liste de mots), retourne la liste des 10 meilleurs documents (docID, score), classés par score décroissant, en appliquant la similarité cosinus.

    Dans l'exemple ci-dessous. C'est la mesure de Jaccard qui est utilisée.
    """
    scores = [0.0 for _ in range(len(self.docs))]
    query_weights = {term: 1 + np.log10(count)
                     for term, count in Counter(query).items()}
    query_set = set(query)
    
    for d in range(len(self.docs)):
        # Mots réellement présents dans le document d (tf > 0)
        doc_terms = self.tf[d]
        #Norme L2 du document
        norm = np.sqrt(sum(self.tfidf[d][term] ** 2 for term in doc_terms))
        if norm == 0:
            continue

        #Produit scalaire requete x document
        dot_product = sum(w_q * self.tfidf[d][term]
                          for term, w_q in query_weights.items()
                          if term in doc_terms)
        # Calcul du score cosinus
        scores[d] = dot_product/norm

    # Tri des scores
    ranking = [idx for idx, sim in sorted(enumerate(scores),
                                        key=lambda pair: pair[1], reverse=True)]
    results = []
    for i in range(10):
        results.append((ranking[i], scores[ranking[i]]))
    return results

# Ne pas modifier les lignes ci-dessous
IRSystem.rank_retrieve = rank_retrieve
run_tests(ir_system, part=3)

===== Running tests =====
Cosine Similarity Test
    Score: 3 Feedback: 5/5 Correct. Accuracy: 1.000000
